# Student Performance Factors: Hypothesis Testing

## Purpose

This notebook tests the project's three hypotheses using the cleaned
Student Performance Factors dataset.

The analysis focuses on:

1. Attendance and exam scores, accounting for study hours and previous scores.
2. Whether the association between tutoring and exam scores varies with
   previous scores.
3. Differences in exam scores across resource-access groups, accounting
   for other factors specified in the hypothesis plan.

## Approach

Each test will include:

- The research question and null and alternative hypotheses.
- The chosen method and why it is appropriate.
- Checks of the model assumptions.
- Results, including effect estimates and 95% confidence intervals.
- An interpretation linked to the research question and its limitations.

A significance level of 0.05 will be used, with Holm correction applied
to the three main hypothesis tests to account for multiple testing.

The results describe associations and do not establish causation.

## Import Libraries

These libraries support data loading, regression analysis, diagnostic
plots and adjustment for multiple hypothesis tests.

In [2]:
# File paths
from pathlib import Path

# Data handling and numerical calculations
import pandas as pd
import numpy as np

# Visualisations and diagnostic plots
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Statistical modelling and multiple-testing correction
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

## Load the Cleaned Dataset

The cleaned dataset is loaded from the project's data/processed folder.
The first five rows and dataset dimensions are displayed to confirm
that it has loaded correctly.

In [3]:
# Locate the cleaned dataset
data_path = (
    Path("..")
    / "data"
    / "processed"
    / "StudentPerformanceFactors_cleaned.csv"
)

# Load the data
df = pd.read_csv(data_path)

# Check the number of rows and columns
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# Preview the first five rows
df.head()

Rows: 6,607
Columns: 20


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


## Hypothesis 1: Attendance and Exam Scores

### Research Question

Is attendance associated with exam scores after accounting for
study hours and previous scores?

### Null Hypothesis (H₀)

Attendance has no linear association with exam scores after accounting
for study hours and previous scores.

The attendance coefficient in the regression model equals zero.

### Alternative Hypothesis (H₁)

Attendance has a linear association with exam scores after accounting
for study hours and previous scores.

The attendance coefficient in the regression model is not zero.

### Method and Justification

Multiple linear regression will be used because exam score is numerical
and the model can examine attendance, study hours and previous scores
together.

- Outcome: Exam_Score
- Main predictor: Attendance
- Adjustment variables: Hours_Studied and Previous_Scores

The attendance coefficient estimates the difference in expected exam
score associated with a one-percentage-point increase in attendance,
holding study hours and previous scores constant.

This is a two-sided test. Model assumptions and influential observations
will be checked before drawing conclusions. The attendance p-value will
be included in the Holm correction across the three main hypothesis tests.

This analysis does not establish causation.

### Check the Variables

Before fitting the model, check that the four required columns are
numerical and contain no missing values. Review their ranges to identify
any values that need consideration when interpreting the results.

In [4]:
# Created with ChatGPT assistance for this project's analysis.

# Select the variables needed for Hypothesis 1
h1_columns = [
    "Exam_Score",
    "Attendance",
    "Hours_Studied",
    "Previous_Scores"
]

# Summarise data types, missing values and ranges
h1_checks = pd.DataFrame({
    "Data type": df[h1_columns].dtypes,
    "Missing values": df[h1_columns].isna().sum(),
    "Minimum": df[h1_columns].min(),
    "Maximum": df[h1_columns].max()
})

h1_checks

,Data type,Missing values,Minimum,Maximum
Exam_Score,int64,0,55,101
Attendance,int64,0,60,100
Hours_Studied,int64,0,1,44
Previous_Scores,int64,0,50,100


### Variable Check Findings

- All four variables are stored as integers and have no missing values.
- Attendance ranges from 60% to 100%.
- Hours studied range from 1 to 44.
- Previous scores range from 50 to 100.
- Exam scores range from 55 to 101.

The exam score of 101 remains an unresolved data-quality concern.
Following the cleaning decision, it will be retained in the main model.
A sensitivity analysis excluding scores above 100 will check whether
this decision affects the results.